In [1]:
import numpy as np 
import pandas as pd

In [3]:
df = pd.read_csv(r"C:\Users\admin\Desktop\RAG-\data\processed\pharmaly_products_rag_ready.csv")


In [7]:
import pandas as pd
import chromadb
import hashlib
from pathlib import Path
from sentence_transformers import SentenceTransformer

LOAD THE EMBEDDING MODEL

In [8]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"

print("Loading embedding model...")

model = SentenceTransformer(MODEL_NAME)

print("Embedding model loaded!")
print("Embedding dimensions:", model.get_sentence_embedding_dimension())

Loading embedding model...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

c:\Users\admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\admin\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

... (7 outputs left out)

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded!
Embedding dimensions: 384


C:\Users\admin\AppData\Local\Temp\ipykernel_16088\2499674318.py:10: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimensions:", model.get_sentence_embedding_dimension())


Create the product documents

In [10]:
def create_product_document(row):
    return f"""
Product: {row['product_name']}
Brand: {row['brand']}
Category: {row['categories']}
Current price: {row['current_price']}
Description: {row['description']}
"""

In [11]:
documents = df.apply(create_product_document, axis=1).tolist()

print("Number of documents:", len(documents))
print("\nExample document:\n")
print(documents[0])

Number of documents: 9148

Example document:


Product: Cheks Postbiotic Balance Bt/45
Brand: Cheks
Category: Sante
Current price: 2700.0
Description: Le Cheks Postbiotic Balance est un complément alimentaire en gummies conçu pour soutenir l’équilibre du microbiote et le confort digestif. Sa formule associe un lysat postbiotique de Lactobacillus plantarum, du butyrate de calcium, de l’inuline ainsi que des extraits de gingembre et de menthe. Cette synergie accompagne la digestion, la régularité du transit et aide à limiter les sensations de ballonnement après les repas. Le Cheks Postbiotic Balance contient également du zinc et de la vitamine D3, qui contribuent au fonctionnement normal du système immunitaire. Ses gummies vegan sont sans sucre ajouté, sans gélatine et sans gluten.



In [12]:
print("Number of documents:", len(documents))
print("Number of rows in CSV:", len(df))
print("First document:\n", documents[0])
print("Last document:\n", documents[-1])

Number of documents: 9148
Number of rows in CSV: 9148
First document:
 
Product: Cheks Postbiotic Balance Bt/45
Brand: Cheks
Category: Sante
Current price: 2700.0
Description: Le Cheks Postbiotic Balance est un complément alimentaire en gummies conçu pour soutenir l’équilibre du microbiote et le confort digestif. Sa formule associe un lysat postbiotique de Lactobacillus plantarum, du butyrate de calcium, de l’inuline ainsi que des extraits de gingembre et de menthe. Cette synergie accompagne la digestion, la régularité du transit et aide à limiter les sensations de ballonnement après les repas. Le Cheks Postbiotic Balance contient également du zinc et de la vitamine D3, qui contribuent au fonctionnement normal du système immunitaire. Ses gummies vegan sont sans sucre ajouté, sans gélatine et sans gluten.

Last document:
 
Product: Gelée Gommante Douceur
Brand: Avène
Category: Visage
Current price: nan
Description: La Gelée Gommante Douceur Avène exfolie la peau avec un fini lisse et lu

In [13]:
embeddings = model.encode(
    documents,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Embeddings shape:", embeddings.shape)
print("Number of embeddings:", len(embeddings))
print("Dimensions per embedding:", len(embeddings[0]))

Batches:   0%|          | 0/286 [00:00<?, ?it/s]

Embeddings shape: (9148, 384)
Number of embeddings: 9148
Dimensions per embedding: 384


In [15]:
def get_embeddings(texts):
    """
    Convert a list of texts into embeddings.
    Returns a list of numerical vectors.
    """
    if not texts:
        return []

    embeddings = model.encode(
        texts,
        batch_size=32,
        show_progress_bar=False,
        normalize_embeddings=True
    )

    return embeddings.tolist()
def get_embedding(text):
    """
    Convert one text into one embedding.
    Useful later for Member 3 when embedding user questions.
    """

    return get_embeddings([text])[0]


if __name__ == "__main__":
    example = "Vitamin C supplement for immunity"

    vector = get_embedding(example)

    print("Embedding generated successfully!")
    print("Vector dimensions:", len(vector))
    print("First 5 values:", vector[:5])

Embedding generated successfully!
Vector dimensions: 384
First 5 values: [-0.090377077460289, 0.007073534186929464, -0.052637532353401184, -0.0326639860868454, 0.019001878798007965]


Create and manage our persistent ChromaDB database.

In [16]:
import chromadb

# 1. Define database location
DB_PATH = "./chroma_db"

# 2. Create a persistent ChromaDB client
client = chromadb.PersistentClient(
    path=DB_PATH
)

# 3. Create or access our collection
collection = client.get_or_create_collection(
    name="pharmaly_products",
    metadata={"hnsw:space": "cosine"}
)


# 4. Function to store products
def add_documents(ids, documents, embeddings, metadatas):

    collection.upsert(
        ids=ids,
        documents=documents,
        embeddings=embeddings,
        metadatas=metadatas
    )


# 5. Function to count stored products
def get_collection_count():

    return collection.count()


# 6. Display database information
print("Connected to ChromaDB!")
print("Collection:", collection.name)
print("Number of stored products:", get_collection_count())

Connected to ChromaDB!
Collection: pharmaly_products
Number of stored products: 0


embadding of documents

In [17]:
ids = [str(i) for i in range(len(documents))]

In [18]:
print(df.columns.tolist())
df.head(3)

['product_name', 'brand', 'categories', 'original_price', 'current_price', 'description', 'product_url']


,product_name,brand,categories,original_price,current_price,description,product_url
0,Cheks Postbiotic Balance Bt/45,Cheks,Sante,NaN,2700.0,Le Cheks Postbiotic Balance est un complément ...,https://www.pharmaly-dz.com/produit/cheks-post...
1,Acm Novophane Complément Alimentaire Ongles/ch...,Laboratoires ACM,Sante,NaN,6500.0,La crème apaisante ACM Sédacalm 120ml est spéc...,https://www.pharmaly-dz.com/produit/acm-novoph...
2,Granions Biotine Vitamine B8 46g,Granions,Sante,NaN,4800.0,Granions Biotine Vitamine B8 est un complément...,https://www.pharmaly-dz.com/produit/granions-b...


metadata

In [19]:
import pandas as pd

metadatas = []

for _, row in df.iterrows():

    metadata = {
        "product_name": str(row["product_name"]),
        "brand": str(row["brand"]) if pd.notna(row["brand"]) else "Not specified",
        "category": str(row["categories"]) if pd.notna(row["categories"]) else "Not specified",
        "product_url": str(row["product_url"]) if pd.notna(row["product_url"]) else ""
    }

    if pd.notna(row["current_price"]):
        metadata["current_price"] = float(row["current_price"])

    if pd.notna(row["original_price"]):
        metadata["original_price"] = float(row["original_price"])

    metadatas.append(metadata)

print("Number of metadata:", len(metadatas))

Number of metadata: 9148


Stocker dans chroma database

In [20]:
ids = [str(i) for i in range(len(documents))]

batch_size = 1000

for start in range(0, len(documents), batch_size):

    end = min(start + batch_size, len(documents))

    collection.upsert(
        ids=ids[start:end],
        documents=documents[start:end],
        embeddings=embeddings[start:end].tolist(),
        metadatas=metadatas[start:end]
    )

    print(f"Stored: {end}/{len(documents)}")

print("Total products:", collection.count())

Stored: 1000/9148
Stored: 2000/9148
Stored: 3000/9148
Stored: 4000/9148
Stored: 5000/9148
Stored: 6000/9148
Stored: 7000/9148
Stored: 8000/9148
Stored: 9000/9148
Stored: 9148/9148
Total products: 9148


In [25]:
# Retrieve one product, including its stored embedding
stored = collection.get(
    ids=["0"],
    include=["documents", "metadatas", "embeddings"]
)

# Extract the stored embedding
stored_embedding = stored["embeddings"][0]

print("Database count:", collection.count())
print("Stored embedding dimensions:", len(stored_embedding))
print("Embedding norm:", np.linalg.norm(stored_embedding))

print("\nStored product:")
print(stored["metadatas"][0]["product_name"])
print("Product URL:", stored["metadatas"][0]["product_url"])

Database count: 9148
Stored embedding dimensions: 384
Embedding norm: 0.9999999583564362

Stored product:
Cheks Postbiotic Balance Bt/45
Product URL: https://www.pharmaly-dz.com/produit/cheks-postbiotic-balance-bt-45/
